# Nivel 2: ¿qué tributario trae el sedimento?
Abrir este notebook desde la carpeta del kit (la que tiene `nivel1/` y `nivel2/`). Correr las celdas en orden.

In [ ]:
# carga de datos
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score as pr_auc, precision_recall_curve

BASE = next(
    (
        candidate
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "maipo-starter-kit")
        if (candidate / "nivel2" / "nivel2_tributarios.csv").is_file()
    ),
    None,
)
if BASE is None:
    raise FileNotFoundError(
        "No se encontró la carpeta del kit con nivel2/nivel2_tributarios.csv. "
        "Abre el notebook desde una carpeta dentro del kit."
    )

n1 = pd.read_csv(BASE / "nivel1" / "nivel1_manzano.csv", parse_dates=["fecha"])
n2 = pd.read_csv(BASE / "nivel2" / "nivel2_tributarios.csv", parse_dates=["fecha"])
df = n1.merge(n2, on="fecha")
sub = pd.read_csv(BASE / "nivel2" / "subcuencas.csv").set_index("subcuenca_id")


def carga(q, ssc):
    """Carga de sedimento en t/día (q en m3/s, ssc en mg/L)."""
    return 0.0864 * q * ssc

df.shape

## Meta 1: el Colorado aporta ~la mitad del sedimento con ~un cuarto del agua
Solo días con sedimento y caudal medidos en El Manzano **y** en el Colorado.

In [ ]:
# meta 1
m = df.dropna(subset=["q_m3s", "ssc_mgL", "q_colorado_m3s", "ssc_colorado_mgL"])
frac_agua = m.q_colorado_m3s.sum() / m.q_m3s.sum()
frac_sed = carga(m.q_colorado_m3s, m.ssc_colorado_mgL).sum() / carga(m.q_m3s, m.ssc_mgL).sum()
print(f"Días en común: {len(m)} ({m.fecha.min().date()} a {m.fecha.max().date()})")
print(f"Colorado aporta {frac_agua:.1%} del agua y {frac_sed:.1%} del sedimento")

In [ ]:
# meta 1: gráfico
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Agua", "Sedimento"]
colorado = [frac_agua * 100, frac_sed * 100]
resto = [100 - value for value in colorado]
positions = range(len(labels))
ax.bar(positions, colorado, color="#B5651D", label="Colorado")
ax.bar(positions, resto, bottom=colorado, color="#4C8BF5", label="Resto del Maipo")
for x, colorado_value, resto_value in zip(positions, colorado, resto):
    ax.text(x, colorado_value / 2, f"Colorado\n{colorado_value:.1f}%", ha="center", va="center", color="white")
    ax.text(x, colorado_value + resto_value / 2, f"Resto del Maipo\n{resto_value:.1f}%", ha="center", va="center", color="white")
ax.set_xticks(list(positions), labels)
ax.set_ylabel("% de lo que llega a El Manzano")
ax.set_ylim(0, 100)
ax.set_title(f"Aporte del río Colorado ({len(m)} días comunes)")
ax.legend()
plt.show()


## Meta 2: rendimiento (t/km²/año) de las 3 estaciones con sedimento
Se calcula con todos los días de cada estación y con los días comunes a las tres (para discutir la incertidumbre).

In [ ]:
# meta 2
EST = {  # subcuenca_id -> (caudal, sedimento)
    "manzano": ("q_m3s", "ssc_mgL"),
    "colorado_maipo": ("q_colorado_m3s", "ssc_colorado_mgL"),
    "san_alfonso": ("q_san_alfonso_m3s", "ssc_san_alfonso_mgL"),
}
comunes = df.dropna(subset=[c for par in EST.values() for c in par])
filas = []
for sid, (q, s) in EST.items():
    d = df.dropna(subset=[q, s])
    area = sub.loc[sid, "area_km2"]
    filas.append({
        "estacion": sid,
        "dias_ssc": len(d),
        "rend_todos_los_dias": carga(d[q], d[s]).mean() * 365 / area,
        "rend_dias_comunes": carga(comunes[q], comunes[s]).mean() * 365 / area,
    })
atrib = ["pend_media_deg", "pct_glaciar", "pct_area_sobre_4000m"]
rend = (pd.DataFrame(filas).set_index("estacion").join(sub[atrib])
        .sort_values("rend_todos_los_dias", ascending=False).round(1))
print(f"Días comunes a las 3 estaciones: {len(comunes)}")
rend

In [ ]:
# meta 2: gráfico
estaciones = list(rend.index)
positions = list(range(len(estaciones)))
width = 0.36
colores_estacion = ["#B5651D" if sid == "colorado_maipo" else "#4C8BF5" for sid in estaciones]
ax = plt.gca()
ax.clear()
ax.bar([i - width / 2 for i in positions], rend["rend_todos_los_dias"], width=width,
       color=colores_estacion, label="Todos los días")
ax.bar([i + width / 2 for i in positions], rend["rend_dias_comunes"], width=width,
       color="#D3D3D3", edgecolor="#808080", hatch="//", label="Días comunes")
ax.set_xticks(positions, estaciones, rotation=0)
ax.set_ylabel("t/km²/año")
ax.set_title("Rendimiento de sedimento por estación")
ax.legend()
plt.show()


## Meta 3: ranking de subcuencas sin sedimento medido
Colorado vs San Alfonso: la pendiente es casi igual; lo que cambia es el % del área sobre 4.000 m (y el % glaciar). Se ordena por esos dos atributos.

In [ ]:
# meta 3
# ponytail: ranking por altura + desempate glaciar; pasar a un índice ponderado
# solo si aparecen más estaciones con SSC para calibrar los pesos.
cols_rank = ["drena_a", "area_km2", "elev_media_m", *atrib, "geologia"]
ranking = (sub[sub.tiene_ssc == 0]
           .sort_values(["pct_area_sobre_4000m", "pct_glaciar"], ascending=False)[cols_rank])
ranking.insert(0, "rank", range(1, len(ranking) + 1))
ranking

In [ ]:
# meta 3: gráfico (% sobre 4000 m y % glaciar de cada subcuenca sin SSC)
ax = ranking[["pct_area_sobre_4000m", "pct_glaciar"]].iloc[::-1].plot.barh(
    figsize=(7, 5), color=["#4C8BF5", "#A0A0A0"])
ax.set_xlabel("% del área")
ax.set_title("Ranking de subcuencas sin sedimento medido (arriba = más aporte esperado)")
ax.legend(["% sobre 4.000 m", "% glaciar"])
plt.show()


In [ ]:
# meta 2 y 3: rendimiento vs altura
CAFE, AZUL, GRIS = "#B5651D", "#4C8BF5", "#888888"
color = lambda sid, drena: CAFE if "colorado" in (sid + str(drena)) else (GRIS if drena == "manzano" else AZUL)

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(9, 8.5), sharex=True,
                              gridspec_kw={"height_ratios": [3, 2.6], "hspace": 0.15})

# arriba: estaciones medidas + tendencia
x, y = rend.pct_area_sobre_4000m, rend.rend_todos_los_dias
ax.plot(x.sort_values(), y[x.sort_values().index], ls="--", color=GRIS, lw=1, zorder=1)
for sid in rend.index:
    ax.scatter(x[sid], y[sid], s=160, color=color(sid, ""), edgecolor="white", lw=1.5, zorder=3)
    ax.annotate(f"{sid}\n{y[sid]:.0f} t/km²/año", (x[sid], y[sid]), xytext=(10, -4),
                textcoords="offset points", fontsize=9, va="center")
ax.axvspan(x.max(), 42, color=CAFE, alpha=0.07)
ax.text(41.5, y.min(), "más alto que el Colorado\n→ mayor aporte esperado", ha="right",
        fontsize=9, color=CAFE, style="italic")
ax.set_ylim(180, 880)
ax.set_ylabel("Rendimiento medido\n(t/km²/año)")
ax.set_title("Más área sobre 4.000 m → más sedimento por km²", loc="left", fontsize=12, weight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# abajo: subcuencas sin medición, una fila por subcuenca (ordenadas por ranking)
filas = ranking.iloc[::-1]
for j, (sid, r) in enumerate(filas.iterrows()):
    c = color(sid, r.drena_a)
    ax2.hlines(j, 0, r.pct_area_sobre_4000m, color=c, lw=2, alpha=0.6)
    ax2.scatter(r.pct_area_sobre_4000m, j, s=60, color=c, zorder=3)
ax2.set_yticks(range(len(filas)), [f"#{r} {s}" for s, r in filas["rank"].items()], fontsize=9)
ax2.axvspan(x.max(), 42, color=CAFE, alpha=0.07)
ax2.set_ylim(-0.6, len(filas) - 0.4)
ax2.spines[["top", "right"]].set_visible(False)
ax2.set_title("Subcuencas sin sedimento medido (ranking propuesto)", loc="left", fontsize=10)
ax2.set_xlabel("% del área de la subcuenca sobre 4.000 m")
ax2.set_xlim(-2, 42)
plt.show()

In [ ]:
# meta 3: trampa de la guía, caudal por km2 subestimado por captaciones hidroeléctricas
# NO usar para rankear
esp = pd.Series({sid: df[col].mean() / sub.loc[sid, "area_km2"] * 1000 for col, sid in [
    ("q_olivares_m3s", "olivares"), ("q_colorado_alto_m3s", "colorado_alto"),
    ("q_volcan_m3s", "volcan"), ("q_colorado_m3s", "colorado_maipo"),
    ("q_san_alfonso_m3s", "san_alfonso")]})
ax = plt.gca()
ax.clear()
captadas = ["olivares", "colorado_alto", "volcan"]
ax.bar(range(3), esp.loc[captadas], color="#D3D3D3", edgecolor="#808080", hatch="//",
       label="afectada por captación hidroeléctrica")
ax.bar(3, esp.loc["colorado_maipo"], color="#B5651D", label="Colorado")
ax.bar(4, esp.loc["san_alfonso"], color="#4C8BF5", label="Maipo / San Alfonso")
ax.set_xticks(range(len(esp)), esp.index, rotation=0)
ax.set_ylabel("L/s/km²")
ax.set_title("Caudal específico (afectado por captaciones de centrales)")
ax.legend()
plt.show()

## Meta 4: modelo de ML vs regla `p_liq_48h_mm` (PR-AUC, prueba `anio_hidro` 2013–2018)
Objetivo: 1 si `ssc_mgL` supera el percentil 95 (calculado solo con entrenamiento). Sin columnas de sedimento ni de eventos en X.

In [ ]:
# meta 4: datos y partición temporal
FEATS = ["q_m3s", "q_colorado_m3s", "q_volcan_m3s", "q_olivares_m3s", "q_colorado_alto_m3s",
         "p_liq_mm", "p_liq_48h_mm", "p_acum3d_mm", "p_liq_colorado_mm", "p_liq_san_alfonso_mm",
         "iso0_m", "frac_bajo_iso0", "t_media_C", "mes"]
t = df[df.ssc_mgL.notna()]
# año hidrológico abr–mar: un mismo verano (dic–mar) no se parte entre train y test
anio = t.anio_hidro
train, test = t[anio <= 2012], t[(anio >= 2013) & (anio <= 2018)]
umbral = train.ssc_mgL.quantile(0.95)
y_tr, y_te = (train.ssc_mgL > umbral).astype(int), (test.ssc_mgL > umbral).astype(int)
print(f"Umbral p95 (train): {umbral:.0f} mg/L | positivos train {y_tr.sum()}/{len(y_tr)}, "
      f"test {y_te.sum()}/{len(y_te)} (prevalencia {y_te.mean():.3f})")

In [ ]:
# meta 4: entrenamiento y PR-AUC
# min_samples_leaf=5: hojas con al menos 5 días, para no memorizar días sueltos (solo ~5% son eventos)
modelo = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=0, n_jobs=-1)
modelo.fit(train[FEATS], y_tr)  # sklearn >= 1.4 acepta NaN
prob = modelo.predict_proba(test[FEATS])[:, 1]
regla = test.p_liq_48h_mm.fillna(0)
print(f"PR-AUC modelo: {pr_auc(y_te, prob):.3f} | regla p_liq_48h_mm: {pr_auc(y_te, regla):.3f}")

In [ ]:
# meta 4: gráfico curva precisión-recall
fig, ax = plt.subplots(figsize=(6, 5))
for nombre, score, color in [("Modelo", prob, "#4C8BF5"), ("Regla p_liq_48h_mm", regla, "gray")]:
    p, r, _ = precision_recall_curve(y_te, score)
    ax.plot(r, p, color=color, label=f"{nombre} (PR-AUC {pr_auc(y_te, score):.3f})")
ax.axhline(y_te.mean(), ls="--", color="gray", label="Azar (prevalencia)")
ax.set_xlabel("Recall")
ax.set_ylabel("Precisión")
ax.set_title("Curva precisión-recall (prueba anio_hidro 2013–2018)")
ax.legend()
plt.show()


In [ ]:
# meta 4: verano vs invierno (año completo: verano oct–mar, invierno abr–sep)
TEMPORADAS = {"verano (oct–mar)": [10, 11, 12, 1, 2, 3], "invierno (abr–sep)": [4, 5, 6, 7, 8, 9]}
filas = []
for temporada, meses in TEMPORADAS.items():
    dias = test.mes.isin(meses).to_numpy()
    filas.append({"temporada": temporada, "modelo": pr_auc(y_te[dias], prob[dias]),
                  "regla": pr_auc(y_te[dias], regla[dias]), "azar": y_te[dias].mean(),
                  "eventos": y_te[dias].sum(), "dias": dias.sum()})
estac = pd.DataFrame(filas).set_index("temporada")
print(estac.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
estac[["modelo", "regla"]].plot.bar(ax=ax, rot=0, color=["#4C8BF5", "gray"])
ax.set_xticks(range(len(estac)), [f"{t}\n({e} eventos)" for t, e in estac["eventos"].items()])
for i, azar in enumerate(estac["azar"]):
    ax.hlines(azar, i - 0.35, i + 0.35, color="black", linestyle="--", linewidth=1.5,
              label="Azar (prevalencia)" if i == 0 else None)
ax.set_ylabel("PR-AUC")
ax.set_title("Desempeño por temporada (prueba anio_hidro 2013–2018)")
ax.legend()
plt.show()


In [ ]:
# meta 4: importancia de variables
imp = permutation_importance(modelo, test[FEATS], y_te, scoring="average_precision",
                             n_repeats=10, random_state=0)
imp_s = pd.Series(imp.importances_mean, FEATS).sort_values()
imp_std_s = pd.Series(imp.importances_std, index=FEATS).loc[imp_s.index]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(imp_s.index, imp_s.values, xerr=imp_std_s.values, color="#4C8BF5",
        ecolor="gray", capsize=3)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Caída de PR-AUC al permutar")
ax.set_title("Importancia de variables")
plt.show()


In [ ]:
# meta 4: eventos por mes (por qué la regla de lluvia falla)
riesgo = test.assign(evento=y_te).groupby("mes").evento.mean() * 100
meses_es = {1: "ene", 2: "feb", 3: "mar", 4: "abr", 5: "may", 6: "jun",
            7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov", 12: "dic"}
colores = ["#B5651D" if mes in (12, 1, 2, 3) else "#4C8BF5" for mes in riesgo.index]
ax = riesgo.plot.bar(figsize=(7, 3), rot=0, color=colores)
ax.set_xticks(range(len(riesgo)), [meses_es[mes] for mes in riesgo.index])
ax.set_ylabel("% de días > p95")
ax.set_title("Riesgo de sedimento extremo por mes (anio_hidro 2013–2018)")
plt.show()


In [ ]:
# chequeo: reglas de /auditar (sin fuga, partición temporal honesta)
assert not [c for c in FEATS if c.startswith(("ssc", "evento"))], "fuga de sedimento/eventos en X"
assert train.anio_hidro.max() <= 2012 and test.anio_hidro.between(2013, 2018).all()
assert not set(train.fecha) & set(test.fecha), "días repetidos entre train y test"
print("Chequeos de auditoría OK")

## Conclusiones de la Meta 4 (Random Forest, partición por `anio_hidro`: train <= 2012, test 2013–2018)

- **Global:** PR-AUC modelo 0.127 vs regla `p_liq_48h_mm` 0.035, con azar (prevalencia) 0.041. El modelo da ~3× el azar.
- **Partición honesta:** antes se cortaba por año calendario y el verano 2012–13 (dic en train, ene–mar en test) quedaba partido. Días vecinos del mismo deshielo están muy correlacionados, así que el modelo "veía" parte del test. Con `anio_hidro` cada verano queda entero de un lado.
- **Verano (oct–mar):** modelo 0.138 vs regla 0.069 (azar 0.079). Aquí está toda la ventaja: los pulsos de deshielo dependen del calendario, la temperatura y el caudal (`mes`, `t_media_C` y `q_m3s` son las variables más importantes), no de la lluvia.
- **Invierno (abr–sep):** modelo 0.011 **pierde** contra la regla (0.017), con solo 9 eventos (azar 0.009). Con tan pocos positivos el PR-AUC es muy ruidoso, y además la muestra de las 08:00 h (`ssc_hora_h`) se pierde los picos breves de las tormentas cálidas. Para invierno la regla física sigue siendo la mejor referencia.

### Elección del modelo: Random Forest
Se usa `RandomForestClassifier(n_estimators=500, min_samples_leaf=5)` por indicación de los organizadores. Como referencia, con la misma partición `HistGradientBoostingClassifier` daba 0.078. `class_weight="balanced"` no ayuda: PR-AUC solo mide el orden de las probabilidades.

Ojo: esa comparación entre algoritmos se hizo mirando el test, así que 0.127 es algo optimista. Para estimarlo con honestidad habría que validar con `GroupKFold` por `anio_hidro` dentro del train.
